# Гибридный поиск

## BM25

In [ ]:
# Импортируем BM25Okapi — класс для вычисления BM25, популярного метода лексического поиска
from rank_bm25 import BM25Okapi
import re
import numpy as np

docs = []

# создаём список текстов
corpus = [doc.page_content for doc in docs]

# Индексируем исходные объекты Document в словарь
id2doc = {d.metadata.get("id", str(i)): d for i, d in enumerate(docs)}

# Определяем регулярное выражение для токенизации текста
TOKEN_RE = re.compile(r"[A-Za-zА-Яа-я0-9_]+", re.UNICODE)

# Функция токенизации: разбиваем строку на токены и приводим к нижнему регистру
def tokenize(txt):
    return [t.lower() for t in TOKEN_RE.findall(txt)]

# Применяем токенизацию к каждому документу в корпусе
tokenized_corpus = [tokenize(d) for d in corpus]  

# Создаём объект BM25Okapi на основе токенизированного корпуса
# bm25 будет использоваться для вычисления релевантности документов запросу
bm25 = BM25Okapi(tokenized_corpus)

def bm25_candidates(query, k):
    toks = tokenize(query) # токенизируем вопрос
    scores = bm25.get_scores(toks)  # получаем скор
    # Сортируем индексы документов по убыванию скоринга и выбираем top-k
    top_idx = np.argsort(scores)[::-1][:k]
    # Формируем словарь результатов: doc_id → BM25-score
    res = {}
    for i in top_idx:
        doc_id = list(id2doc.keys())[i]
        res[doc_id] = float(scores[i])
    return res

#### Получение скора из векторного хранилища
В langchain метод similarity_search_with_score() у векторного хранилища возвращает Tuple с документом и расстоянием типа float. Важно понимать, что чем меньше это расстояние, тем релевантнее документ. Поэтому нам нужно это значение инвертировать для нормализации этого скора к диапазону [0,1], где 1 это наиболее релевантный, а 0 соответственно наименее.

In [ ]:
from typing import Dict
import numpy as np

def vector_candidates(query, k):
    # Выполняем векторный поиск: получаем пары (документ, расстояние)
    # similarity_search_with_score возвращает dist — чем меньше, тем ближе документ
    docs_scores = vector_store.similarity_search_with_score(query, k=k)

    # Конвертируем расстояние в псевдо-скор: 1 / (1 + dist)
    # Чем меньше dist → тем больше score. Значения будут в (0..1]
    tmp = {d.metadata.get("id"): 1.0 / (1.0 + float(dist)) for d, dist in docs_scores}

    # Нормализуем все скоры в диапазон [0..1], чтобы можно было
    # далее безопасно комбинировать их с BM25
    return normalize(tmp)


def normalize(scores, reverse=False):
    if not scores:
        return {}
    vals = np.array(list(scores.values()), dtype=float)
    if reverse:
        vals = -vals
    lo, hi = float(vals.min()), float(vals.max())
    if hi == lo:
        return {k: 1.0 for k in scores}
    keys = [key for key,_ in scores.items()]
    norm = (vals - lo) / (hi - lo)
    # Собираем обратно в словарь: doc_id → нормализованный скор
    return {key: float(value) for key, value in zip(keys, norm)}

## Reciprocal Rank Fusion (RRF)

Алгоритм RRF — это элегантный способ объединения результатов из разных источников. Он работает с рангами документов, а не с их абсолютными оценками, что делает его устойчивым к различиям в масштабах скоров.

Основная идея: каждому документу присваивается суммарный скор по формуле RRF_score = Σ (1 / (k + rank)) для каждого ранжированного списка. Параметр k (обычно 60) контролирует, насколько быстро падает вклад документов с низкими рангами. Чем выше ранг документа (то есть чем он релевантнее), тем больше его вклад в итоговый скор.

RRF избегает сложной нормализации баллов и эффективно комбинирует результаты от BM25 и векторного поиска, давая каждому методу равные возможности повлиять на финальный рейтинг.

In [ ]:
from typing import Dict
from collections import defaultdict

def to_ranks(scores, higher_better=True):
    items = sorted(scores.items(), key=lambda kv: kv[1], reverse=higher_better)
    return {doc_id: rank+1 for rank, (doc_id, _) in enumerate(items)}

def rrf(bm25, vec, rrf_k=60):
    # превращаем оценки в ранги (1 — лучший)
    ranks_bm25 = to_ranks(bm25, higher_better=True)
    ranks_vec  = to_ranks(vec,  higher_better=True)
    # объединяем носители
    all_ids = set(ranks_bm25) | set(ranks_vec)
    fused = defaultdict(float)
    for doc_id in all_ids:
        if doc_id in ranks_bm25:
            fused[doc_id] += 1.0 / (rrf_k + ranks_bm25[doc_id])
        if doc_id in ranks_vec:
            fused[doc_id] += 1.0 / (rrf_k + ranks_vec[doc_id])
    return dict(fused)